CASO PRATICO: GENERAZIONE TESTO CON LSTM ADDESTRATA SU SHAKESPEARE

Capacità di generare senso dal nulla
Immagina di dare ad un bambino di dare migliaia di pagine scritte da Shakespeare, guarda solo queste pagine e scrivi una pagina come fosse scritta da Shakespeare, costruiamo un immitatore di stili.

- Modellazione a livello di carattere e addestramento su dataset di Shakespeare
- Controllo della creativitàà tramite il tunning della temperatura di campionamento
- Valutazione della coerenza sintattica e analisi dei pattern appresi

La generazione di testo di basa sulla capacità di una rete neurale di prevedere il carattere successivo data una sequenza precedente. Il dataset Shakespeare è un classico per testare la capacità del modello di apprendere stili complessi.

Per allenare il modello, trasformiamo il testo grezzo in una sequenza di numeri interi, dove ogni numero mappa un carattere unico presente nel vocabolario dell'autore.

Architettura per il linguaggio
Componenti essenziali alla pipeline
- Caratterizzazione: l'approccio a livello di carattere permette al modello di imparare non solo le parole, ma anche la punteggiatura e le strutture grammaticali da zero. Il modello impara la probabilità che ad una 'q' segua una 'u', per farlo creiamo un ponte tra testo e numeri con il mapping numerico
- Mapping Numerico: crazione di tabella di traduzione 'char2idx' e 'idx2char' per interfacciare il testo con tensori.
- Input e Target: la rete riceve una sequenza e deve predire la stessa sequenza shiftata di un carattere a destra
- Stateful RNN: mantenimento dello stato tra i batch per apprendere dipendenze che superano la lunghezza della singola sequenza di input. Immagina che la rete abbia un taquino dove si appunta il contesto del discorso, senza questo taquino la rete dimenticherebbe l'inizio della frase.

Il target è sempre il carattere successivo

Preparazione dei dati
Come questi caratteri diventano vettori digeribili dalla macchina
Vettorizzazione: il testo viene suddiviso in segmenti di lunghezza fissa (tensori). Un segmento di input lungo 'L' produrrà un target lungo 'L', dove ogni elemento è il carattere successivo rispetto all'input
- Embedding Layer: un layer di embedding trasforma gli indici dei caratteri in vettori densi che catturano le relazioni semantiche latenti tra i simboli. Non diamo solo un numero ad una lettera, ma posizioniamo le lettere nello spazio in modo che quelle che si comportano in modo simile sono vicine.
- Dimensione del vocabolario: il vocabolario totale del dataset Shakespeare comprende circa 65 caratteri unici, rendendo l'output della rete un vettore di probabilità di dimensione ridotta.

Come misuriamo quanto il nostro poeta elettronico sta sbagliando?

Funzione di Loss
Cross-Entorpy nel tempo
Durante l'addestramento, utiliziamo la 'Sparse Categorical Cross-Entropy' per confrontare la distribuzione di proabilità predetta con il carattere reale.
Questa formula misura quando la mira delle rete sia distante dal bersaglio reale. Durante il training l'obbiettivo è accorciare questa distanza.
L'obbiettivo è minimizzare l'errore cumulativo su tutta la sequenza generata durante l'epoca di trainig.
Se la Loss scende significa che il modello sta smettendo di tirare a caso e sta riconoscendo i pattern

Ma una volta addestrata la rete, come decidiamo quanto deve essere audace nelle sue risposte?

Temperature di campionamento
Regola il bilanciamente tra rigore e follia
Una persona che dice sempre cose scontata è un modello con temperatura 0
Una volta addestrata, la rete produce dei 'logits' per ogni carattere possibile. 
Se scegliessimo sempre il logits più alto la rete sarebbe noiosa e ripetitiva.
Come trasformiamo queste probabilità grezze in una scelta finale senza essere ripetitivi?
La temperatura è un iperparametro applicato durante la fse di inferenza che riscala le probabilità della Softmax, permettendoci di controllare quanto il modello sia conservativo o creativo.
La temperatura è la nostra manopola di creatività
Possiamo decidere se vogliamo l'assistente preciso e rigido o un artista imprevedibile e un po' folle

Meccanica del Campionamento
Dall'output deterministico allo stocastico
E' come lanciare un dado truccato dove le facciate più grandi solo le lettere più probabili.
- Softmax riscalata: modifica della distribuzione di probabilità originale dividendo i logits per il valore delle temperatura
- Campionamento Multinomiale: invece di scegliere sempre il carattere più probabile, estraiamo casualmente in base alla nuova distribuzione
- Diversità: valori di temperatura diversi portano a risultati testuali radicalmente differenti pur partendo dallo stesso modello
- Argmax: corrisponde a una temperatura tendente a zero, dove il modello sceglie sempre e solo l'opzione più sicura, sempre e solo lo stesso risultato.

La temperatura cambia quanto peso diamo alle scelte meno ovvie.

Effetti della Temperatura
Cosa succede concretamente quando giriamo la manopola della temperatura al massimo o al minimo
* Bassa Temperatura (T<0.5): il modello diventa molto sicuro di se. Il testo generato è grammaticalmente corretto ma spesso ripetitivo o banale. Il modello è come uno studente cha ha imparato a memoria la lezione. Sbaglia una virgola ma non inventa nulla.
* Alta Temperatura (T>1.0): le probabilità di appiattiscono. Il modello esplora caratteri improbabili, portando a errori ortografici e alla creazione di parole inesistenti. E' come se il modello avesse bevuto troppo, inizia a balbettare, inventare parole, dimentica di chiudere le parentesi. Rendiamo le valli, cioè le lettere improbabili, alte quasi quanto le vette.
* L'equazione fondamentale: dividendo il logit per la temperatura prima dell'esponenziale, alteriamo drasticamente il peso relativo tra i caratteri più probabili e quelli meno probabili

Ma esiste un punto di equilibrio perfetto?

Lo Sweet Spot
Trovare l'equilibrio stilistico
Per dataset come quello di Shakespeare, una temperatura tra 0.7 e 1.0 solitamente offre il miglior compromesso tra coerenza testuale e variatà espressiva.
In questo range, il modello ha abbastanza corazzio da utilizzare parole ricercate e abbastanza giudizio da non distruggere la sintassi.
Sperimentare con questo valore è fondamentale per evitare che il modello resti bloccato il loop infiniti di punteggiatura o spazi bianchi.

Ma come facciamo a capire se quanto leggiamo è reale o è una illusione

Analisi della coerenza sintattica
Valutare l'intelligenza del testo generato.
Generare testo non significa solo allineare lettere, ma rispetta strutture profonde. 
Come valutare se il modello ha 'capito' le regole del gioco?

La coerenza emerge gradualmente durante le epoche di addestramento, passando dal caos casuale alla parvenza di poesia

Quali sono i segnali che ci dicono che il modello sta davvero capendo?

Indicatori di Qualità
Cosa osservare nel testo generato.
* Struttura delle parole: capacità di aprire e chiudere correttamente le parole e di utilizzare gli spazi in modo sensato. Se una rete mette gli spazi nei punti giusti ha imparato 
* Grammatica locale: accordo tra articoli e sostantivi, coniugazione verbale corretta all'interno di brevi segmenti.
* Sintassi globale: coerenza nei nomi dei personaggi e nella struttura dei dialoghi tipica delle opere teatrali. Se un personaggio inizia a parlare, il modello deve mantenere quel nome fino a quando il modello non cambia
* Punteggiatura: uso corretto di a capo, punti e virgole, per scandire il ritmo del testo.

Non tutto è perfetto, ci sono dei limiti invalicabili per queste architetture così semplici.
Limite a Sfide

Il primo problema principale è la memoria a  breve termine, sappaimo che le RNN classiche rischiano di perdere informazioni se il testo è troppo lungo.

- Assenza di memoria a lungo termine: senza architetture avanzate, il modello potrebbe dimenticare il soggetto della frase dopo poche righe, perdendo la coerenza narrativa.
- Allucinazioni: il modello può inventare perole che 'suonano' come Shakespeare ma non esistono nel dizionario inglese, un fenomeno comune del Deep Learning generativo.
- Valutazione umana: a differenza della classificazione, la generazione di testo richiede spesso un giudizio qualitativo soggettivo per misurare l'efficacia dello stile

Alla fine l'unico vero giudice è l'occhio umano, capace di distinguere la poesia dal rumore statistico.
Questa capacità emerge dal nulla durante l'addestramento.

L'evoluzione dell'apprendimento è quasi magia.
Nelle prime fasi del training, il modello genera rumore. Dopo qualche epoca, inizia a formare parole brevi e spazi. Infine , apprende la struttura del dialogo.

Ell'epoca 1 il modello emette suoni casuali, all'epoca 5 scopre che esiste uno spazio bianco, all'epoca 10 iniza a scrivre parole come di o end
Alla fine delle epoche inizia ad inserire il nome dei personaggi con la maiuscola iniziale

Questa progressione dimostra che la rete estragga gerarchicamente le regole del linguaggio partendo dalla pura osservazione statistica dei caratteri.



In [ ]:
import os
os.environ["KERAS_BACKEND"] = "torch"

import keras
import tensorflow as tf
import numpy as np

# Forza l'uso dei calcoli veloci a 16-bit (ideale per la 7900 XTX)
keras.mixed_precision.set_global_policy("mixed_float16")


# 1. ACQUISIZIONE E PRE-PROCESSAMENTO DEI DATI
# Scarichiamo il dataset "Tiny Shakespeare" direttamente dai server Google
path_to_file = tf.keras.utils.get_file(
    'shakespeare.txt', 
    'https://storage.googleapis.com/download.tensorflow.org/data/shakespeare.txt'
)

# Lettura del testo e decodifica in formato stringa
text = open(path_to_file, 'rb').read().decode(encoding='utf-8')
# Il vocabolario rappresenta l'insieme dei caratteri unici che il modello può "conoscere"
vocab = sorted(set(text))
char2idx = {u:i for i, u in enumerate(vocab)}
idx2char = np.array(vocab)

# Convertiamo l'intero testo in una rappresentazione numerica (vettorizzazione)
# Teoria: La rete non comprende i simboli, ma le relazioni statistiche tra indici numerici
text_as_int = np.array([char2idx[c] for c in text])

# 2. CREAZIONE DEL DATASET CON TF.DATA
# Definiamo la lunghezza massima della sequenza di input (finestra temporale)
seq_length = 250
char_dataset = tf.data.Dataset.from_tensor_slices(text_as_int)

# Creiamo sequenze di lunghezza seq_length + 1 (input + target shiftato)
sequences = char_dataset.batch(seq_length + 1, drop_remainder=True)

def split_input_target(chunk):
    """
    Divide il chunk in input (caratteri da 0 a L-1) e target (da 1 a L)
    Esempio: Input 'Hell' -> Target 'ello'
    """
    input_text = chunk[:-1]
    target_text = chunk[1:]
    return input_text, target_text

dataset = sequences.map(split_input_target)

# Bufferizzazione e batching per ottimizzare l'uso della GPU (Best Practice 2026)
BATCH_SIZE = 128
BUFFER_SIZE = 10000
dataset = dataset.cache().shuffle(BUFFER_SIZE).batch(BATCH_SIZE, drop_remainder=True).prefetch(tf.data.AUTOTUNE)

# 3. DEFINIZIONE DEL MODELLO (ARCHITETTURA RECENTE)
# Usiamo un layer Embedding, un layer GRU (più veloce delle LSTM) e un Dense finale
vocab_size = len(vocab)
embedding_dim = 256
rnn_units = 1024

def build_model(vocab_size, embedding_dim, rnn_units, batch_size):
    model = keras.Sequential([
        # Embedding: trasforma indici discreti in vettori densi in uno spazio continuo
        keras.layers.Embedding(vocab_size, embedding_dim),
        # GRU: gestisce la memoria a lungo termine della sequenza
        # 'stateful=True' permette di ricordare il contesto tra batch successivi durante l'inferenza
        keras.layers.GRU(rnn_units, return_sequences=True, stateful=False, recurrent_initializer='glorot_uniform'),
        # Secondo layer GRU per catturare concetti più complessi
        keras.layers.GRU(rnn_units, return_sequences=True, stateful=False),
        # Dense: produce i 'logits', ovvero i punteggi grezzi per ogni carattere nel vocabolario
        keras.layers.Dense(vocab_size)
    ])
    return model

model = build_model(vocab_size, embedding_dim, rnn_units, BATCH_SIZE)

# 4. TRAINING
# Usiamo 'from_logits=True' perché l'ultimo layer non ha una Softmax (necessaria per il tuning della temperatura)
model.compile(optimizer='adam', loss=keras.losses.SparseCategoricalCrossentropy(from_logits=True))
model.fit(dataset, epochs=50) 

# 5. FUNZIONE DI GENERAZIONE CON TEMPERATURA
import torch # Importa torch per gestire la memoria

def generate_text(model, start_string, temperature=0.7, num_generate=500):
    input_eval = [char2idx[s] for s in start_string]
    input_eval = tf.expand_dims(input_eval, 0)
    text_generated = []

    # DISATTIVA IL CALCOLO DEI GRADIENTI: Risolve l'errore dei 54GB
    with torch.no_grad():
        for i in range(num_generate):
            predictions = model(input_eval)
            predictions = predictions[:, -1, :] / temperature
            
            predicted_id_tensor = keras.random.categorical(predictions, num_samples=1)
            predicted_id = int(keras.ops.convert_to_numpy(predicted_id_tensor)[0, 0])

            # SLIDING WINDOW: Aggiungiamo il nuovo carattere e teniamo solo gli ultimi 250
            new_char_tensor = tf.expand_dims([predicted_id], 0)
            input_eval = tf.concat([input_eval, new_char_tensor], axis=-1)
            
            if input_eval.shape[1] > seq_length:
                input_eval = input_eval[:, 1:]

            text_generated.append(idx2char[predicted_id])

    return (start_string + ''.join(text_generated))

# Esempio di utilizzo (con modello addestrato)
# print(generate_text(model, start_string="ROMEO: ", temperature=0.7))

print(generate_text(model, start_string="ROMEO: ", temperature=0.7))

1115394/1115394 ━━━━━━━━━━━━━━━━━━━━ 1s 1us/step
Epoch 1/50
34/34 ━━━━━━━━━━━━━━━━━━━━ 2483s 72s/step - loss: 3.6560
Epoch 2/50
34/34 ━━━━━━━━━━━━━━━━━━━━ 1803s 53s/step - loss: 2.8548
Epoch 3/50
34/34 ━━━━━━━━━━━━━━━━━━━━ 2672s 80s/step - loss: 2.4231
Epoch 4/50
26/34 ━━━━━━━━━━━━━━━━━━━━ 10:12 77s/step - loss: 2.2731